In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import warnings

from pynwb import NWBHDF5IO

## Inspect NWB file

In [ ]:
nwb_path = r"X:\Personnel\MaryBeth\OpenScope\001709\sub-815059\sub-815059_ses-multiplane-ophys-815059-2025-10-23-11-51-43_ophys.nwb"

with warnings.catch_warnings():
    warnings.filterwarnings("ignore", message="Ignoring cached namespace")
    io = NWBHDF5IO(nwb_path, "r", load_namespaces=True)
    nwb = io.read()

print(nwb)

In [ ]:
plane = 'VISp_6'
mod = nwb.processing[plane]

roi_table = mod['image_segmentation'].plane_segmentations['roi_table']
global_roi_ids = roi_table.id[:]
print(f"global_roi_ids shape: {global_roi_ids.shape}")

image_masks = roi_table['image_mask'][:]
print(f"image_masks shape: {image_masks.shape}")

# Show a single ROI mask
roi_idx = 15 
plt.figure(figsize=(5, 5))
plt.imshow(image_masks[roi_idx], cmap='magma')
plt.title(f"{plane} | ROI index {roi_idx} | roi_id {global_roi_ids[roi_idx]}")
plt.axis('off')
plt.show()

# Show union of all ROI masks
combined_mask = np.max(image_masks, axis=0)
plt.figure(figsize=(5, 5))
plt.imshow(combined_mask, cmap='magma')
plt.title(f"{plane} | combined ROI mask ({len(global_roi_ids)} ROIs)")
plt.axis('off')
plt.show()

In [ ]:
# Inspect RF stimulus table
rf_stim_table = nwb.intervals["receptive_field_block_presentations"].to_dataframe()

rf_stim_table['x_position'] = rf_stim_table['x_position'].astype(float)
rf_stim_table['y_position'] = rf_stim_table['y_position'].astype(float)

xs = np.sort(rf_stim_table.x_position.unique())
ys = np.sort(rf_stim_table.y_position.unique())

print(f"X positions: {xs}")
print(f"Y positions: {ys}")
print(f"Presentations per location: {rf_stim_table.groupby(['x_position','y_position']).size().unique()}")
print(f"Stim duration: {(rf_stim_table.stop_time - rf_stim_table.start_time).mean():.3f} s")

In [ ]:
print(rf_stim_table)

## Look at the different timeseries

In [ ]:
event_timeseries  = mod['event_timeseries']
print(event_timeseries)

In [ ]:
dff_timeseries = mod['dff_timeseries']
print(dff_timeseries)

dff_timeseries_trace = mod['dff_timeseries'].roi_response_series['dff_timeseries'].data[:]
dff_timeseries_timstamps = mod['dff_timeseries'].roi_response_series['dff_timeseries'].timestamps[:]
print(dff_timeseries_trace.shape)
print(dff_timeseries_timstamps.shape)

single_dff_trace = dff_timeseries_trace[:,0]
print(single_dff_trace.shape)
plt.plot(dff_timeseries_timstamps, single_dff_trace)


In [ ]:
raw_timeseries   = mod['raw_timeseries']
print(raw_timeseries)

raw_timeseries_trace = mod['raw_timeseries'].roi_response_series['ROI_fluorescence_timeseries'].data[:]
raw_timeseries_timstamps = mod['raw_timeseries'].roi_response_series['ROI_fluorescence_timeseries'].timestamps[:]
print(raw_timeseries_trace.shape)
print(raw_timeseries_timstamps.shape)

single_raw_trace = raw_timeseries_trace[:,0]
print(single_raw_trace.shape)
plt.plot(raw_timeseries_timstamps, single_raw_trace)


## Get RF for these timeseries

In [ ]:
def get_rf(xs, ys, rf_stim_table, orientation=None, timestamps=None, responses=None,
           t_start=0.05, t_end=0.25):
    
    # create empty array
    unit_rf = np.zeros([ys.size, xs.size])
    # loop over every x position
    for xi, x in enumerate(xs):
        # loop over every y position
        for yi, y in enumerate(ys):
            
            # get all stimulus presentations for this position
            stim_rows = rf_stim_table[(rf_stim_table.x_position == x) & (rf_stim_table.y_position == y)]
            
            if orientation is not None:
                stim_rows = stim_rows[stim_rows.orientation == orientation]
            
            # start with an empty mask for all time points
            all_mask = np.zeros(len(timestamps), dtype=bool)
            
            # loop over each stimulus presentation and build a mask of time points that fall within the response window
            for _, row in stim_rows.iterrows():
                all_mask |= (timestamps >= row.start_time + t_start) & (timestamps < row.start_time + t_end)
            
            # average the df/F values that fall within the response windows marked as true
            unit_rf[yi, xi] = np.mean(responses[all_mask]) if all_mask.any() else np.nan
          
    return unit_rf

In [ ]:
import numpy as np

def get_rf_v2(xs, ys, rf_stim_table, timestamps, responses,
           orientation=None,
           t_start=0.05, t_end=0.25, baseline_window=None):

    unit_rf = np.full((len(ys), len(xs)), np.nan)

    for xi, x in enumerate(xs):
        for yi, y in enumerate(ys):

            stim_rows = rf_stim_table[
                (rf_stim_table.x_position == x) &
                (rf_stim_table.y_position == y)
            ]

            if orientation is not None:
                stim_rows = stim_rows[
                    stim_rows.orientation == orientation
                ]

            trial_responses = []

            for _, row in stim_rows.iterrows():

                response_mask = (
                    (timestamps >= row.start_time + t_start) &
                    (timestamps <  row.start_time + t_end)
                )

                if not response_mask.any():
                    continue

                response_value = np.mean(responses[response_mask])

                # Optional baseline subtraction
                if baseline_window is not None:

                    b0, b1 = baseline_window

                    baseline_mask = (
                        (timestamps >= row.start_time + b0) &
                        (timestamps <  row.start_time + b1)
                    )

                    if baseline_mask.any():
                        baseline = np.mean(responses[baseline_mask])
                        response_value -= baseline

                trial_responses.append(response_value)

            if len(trial_responses) > 0:
                unit_rf[yi, xi] = np.mean(trial_responses)

    return unit_rf

In [ ]:
rf = get_rf_v2(xs, ys, rf_stim_table, orientation=None, timestamps=dff_timeseries_timstamps, responses=single_dff_trace,
            t_start=0.05, t_end=0.25)

plt.imshow(rf)
plt.colorbar()
plt.show()

In [ ]:
rf = get_rf(xs, ys, rf_stim_table, orientation=None, timestamps=raw_timeseries_timstamps, responses=single_raw_trace,
            t_start=0.05, t_end=0.25)

plt.imshow(rf)
plt.colorbar()
plt.show()

## Compute df/f myself

In [ ]:
F0 = np.median(single_raw_trace, axis=0)
dff = (single_raw_trace - F0) / F0

plt.plot(dff)
plt.show()

In [ ]:
plt.plot(single_dff_trace)
plt.show()


## Try OASIS algorithm
#### https://github.com/j-friedrich/OASIS/blob/master/examples/Demo.ipynb

In [ ]:
# trace = single_raw_trace
trace = single_dff_trace

In [ ]:
from oasis.functions import *

c, s, b, g, lam = deconvolve(trace)
plt.plot(s)
plt.show()

print("g:", g)
print("lam:", lam)
print("b:", b)

In [ ]:
# 1. Look at the distribution of nonzero s values
nonzero_s = s[s > 0]
print(f"s.max(): {s.max():.4f}")
print(f"s.min() (nonzero): {nonzero_s.min():.6f}")
print(f"percentiles: {np.percentile(nonzero_s, [50, 75, 90, 95, 99])}")

# 2. Check current spike density
print(f"Fraction of frames with spike: {(s > 0).mean()*100:.1f}%")

In [ ]:
# plot both the raw dF/F trace and the deconvolved activity
plt.plot(trace, label='dF/F')
plt.plot(s, label='deconvolved activity')
plt.legend()
plt.show()

In [ ]:
thresh = 0.1 * s.max()

trace_array = np.asarray(trace, dtype=np.float64)
g = float(g)  # make sure g is a plain Python float / float64 scalar, not float32 or array
thresh = float(0.1 * s.max())

c, s = oasisAR1(trace_array, g=g, s_min=thresh)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))

t = np.arange(len(trace_array))

ax.plot(t, trace_array, label='raw dF/F', color='gray', alpha=0.6, lw=1)
ax.plot(t, c, label='denoised (c)', color='black', lw=1.2)

# mark spike events as vertical ticks/stems at the bottom
spike_times = np.where(s > 0)[0]
spike_amps = s[spike_times]

# scale for visibility - plot spikes as a separate track below the trace
baseline = trace_array.min() - 0.1 * np.ptp(trace_array)
ax.vlines(spike_times, baseline, baseline + spike_amps / spike_amps.max() * 0.1 * np.ptp(trace_array),
           color='red', lw=1, label='inferred spikes')

ax.legend(loc='upper right')
ax.set_xlabel('Frame')
ax.set_ylabel('dF/F')
plt.tight_layout()
plt.show()

In [ ]:
# Zoom into a window around one of the big visible transients, e.g. ~3500-4500
window = slice(3500, 4500)

fig, ax = plt.subplots(figsize=(14, 4))
t_win = np.arange(3500, 4500)

ax.plot(t_win, trace_array[3500:4500], color='gray', alpha=0.6, lw=1, label='raw dF/F')
ax.plot(t_win, c[3500:4500], color='black', lw=1.2, label='denoised (c)')

spk_idx = np.where(s[3500:4500] > 0)[0] + 3500
spk_amps = s[spk_idx]  # fixed: index absolute positions directly into s

baseline = trace_array[3500:4500].min() - 0.05
ax.vlines(spk_idx, baseline, baseline + spk_amps / s.max() * 0.3, color='red', lw=1.5, label='inferred spikes')

ax.legend()
ax.set_xlabel('Frame')
plt.tight_layout()
plt.show()



## Get RF from spikes

In [ ]:
def get_rf_spikes(spike_times, xs, ys, rf_stim_table):
    """
    Calculate receptive field from spike times and stimulus presentations.
    Returns a 2D array (ys x xs) of spike counts in the 0–200ms window after each stimulus.
    """
    unit_rf = np.zeros([ys.size, xs.size])

    for xi, x in enumerate(xs):
        for yi, y in enumerate(ys):
            stim_times = rf_stim_table[
                (rf_stim_table.x_position == str(x)) &
                (rf_stim_table.y_position == str(y))
            ].start_time

            response_spike_count = 0
            for stim_time in stim_times:
                start_idx, end_idx = np.searchsorted(
                    spike_times,
                    [stim_time, stim_time + 0.2]
                )
                response_spike_count += end_idx - start_idx

            unit_rf[yi, xi] = response_spike_count

    return unit_rf

In [ ]:
# Pull the real timestamps out of the NWB object
frame_timestamps = event_timeseries.timestamps[:]  # shape (38455,), seconds

# spk_idx are frame indices where s > 0 (from the FULL trace, not the zoom slice)
spike_times = frame_timestamps[spk_idx]

assert np.all(np.diff(spike_times) >= 0), "timestamps not sorted!"

print(frame_timestamps.min(), frame_timestamps.max())
print(rf_stim_table.start_time.min(), rf_stim_table.start_time.max())

In [ ]:
unit_rf = get_rf_spikes(spike_times, xs, ys, rf_stim_table)

plt.imshow(unit_rf, origin='lower', extent=[xs.min(), xs.max(), ys.min(), ys.max()])
plt.colorbar(label='spike count')
plt.xlabel('x position')
plt.ylabel('y position')
plt.title('Receptive field')
plt.show()

## Test on random units

In [ ]:
from oasis.functions import deconvolve
from oasis.oasis_methods import oasisAR1
from gaussian_filtering import fit_gaussian_to_rf

def get_rf_spikes(spike_times, xs, ys, rf_stim_table, t_start=0.0, t_end=0.2):

    unit_rf = np.zeros([ys.size, xs.size])

    for xi, x in enumerate(xs):
        for yi, y in enumerate(ys):
            stim_times = rf_stim_table[
                (rf_stim_table.x_position == x) &
                (rf_stim_table.y_position == y)
            ].start_time

            response_spike_count = 0
            for stim_time in stim_times:
                start_idx, end_idx = np.searchsorted(
                    spike_times,
                    [stim_time + t_start, stim_time + t_end]
                )
                response_spike_count += end_idx - start_idx

            unit_rf[yi, xi] = response_spike_count

    return unit_rf


def estimate_noise_std(trace):

    return np.median(np.abs(np.diff(trace))) / 0.6745


def run_unit_rf_pipeline(mod, trace_type='dff', roi_idx=None, s_min_nstd=3.0,
                          dff_t_start=0.05, dff_t_end=0.25,
                          spike_t_start=0.0, spike_t_end=0.2,
                          plot=True):
    
    if trace_type == 'raw':
        trace = mod['raw_timeseries'].roi_response_series['ROI_fluorescence_timeseries'].data[:]
        trace_timestamps = mod['raw_timeseries'].roi_response_series['ROI_fluorescence_timeseries'].timestamps[:]
    elif trace_type == 'dff':
        trace = mod['dff_timeseries'].roi_response_series['dff_timeseries'].data[:]
        trace_timestamps = mod['dff_timeseries'].roi_response_series['dff_timeseries'].timestamps[:]
    else:
        raise ValueError(f"Unsupported trace_type: {trace_type}")
    
    n_rois = trace.shape[1]
    if roi_idx is None:
        roi_idx = np.random.randint(n_rois)

    roi_id = global_roi_ids[roi_idx] if 'global_roi_ids' in globals() else roi_idx

    # --- dF/F receptive field ---
    rf_dff = get_rf_v2(xs, ys, rf_stim_table, orientation=None,
                     timestamps=trace_timestamps, responses=trace[:, roi_idx],
                     t_start=dff_t_start, t_end=dff_t_end)
    popt, r_squared_dff, fitted_rf = fit_gaussian_to_rf(rf_dff)

    # --- OASIS deconvolution ---

    trace_array = np.asarray(trace[:, roi_idx], dtype=np.float64)

    c, s, b, g, lam = deconvolve(trace_array, penalty = 0)
    g = float(g)
    noise_std = estimate_noise_std(trace_array)
    s_min = s_min_nstd * noise_std
    c, s = oasisAR1(trace_array, g=g, s_min=s_min)

    spk_idx = np.where(s > 0)[0]
    frame_timestamps = event_timeseries.timestamps[:]
    spike_times = frame_timestamps[spk_idx]

    # --- spike-based receptive field ---
    rf_spikes = get_rf_spikes(spike_times, xs, ys, rf_stim_table,
                               t_start=spike_t_start, t_end=spike_t_end)
    popt, r_squared_spikes, fitted_rf = fit_gaussian_to_rf(rf_spikes)

    if plot:
        fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

        im0 = axes[0].imshow(rf_dff, origin='lower',
                              extent=[xs.min(), xs.max(), ys.min(), ys.max()])
        axes[0].set_title(f'dF/F receptive field\nROI idx {roi_idx}\nR2: {r_squared_dff:.2f}')
        axes[0].set_xlabel('x position')
        axes[0].set_ylabel('y position')
        fig.colorbar(im0, ax=axes[0], label='mean dF/F')

        pct_active = 100 * len(spike_times) / len(s)
        im1 = axes[1].imshow(rf_spikes, origin='lower',
                              extent=[xs.min(), xs.max(), ys.min(), ys.max()])
        axes[1].set_title(f'Spike receptive field\n{len(spike_times)} spikes ({pct_active:.1f}% of frames)\nR2: {r_squared_spikes:.2f}')
        axes[1].set_xlabel('x position')
        axes[1].set_ylabel('y position')
        fig.colorbar(im1, ax=axes[1], label='spike count')

        plt.tight_layout()
        plt.show()

    return {
        'roi_idx': roi_idx, 'roi_id': roi_id,
        'rf_dff': rf_dff, 'rf_spikes': rf_spikes,
        'spike_times': spike_times, 'spk_idx': spk_idx,
        's': s, 'c': c, 'g': g, 's_min': s_min,
        'R2_dff': r_squared_dff,
        'R2_spikes': r_squared_spikes
    }


# Run the receptive field pipeline for a single ROI
result = run_unit_rf_pipeline(mod, trace_type = 'dff') # or 'raw'

In [ ]:
trace_type = 'dff'

trace = mod['raw_timeseries'].roi_response_series['ROI_fluorescence_timeseries'].data[:]
num_rois = trace.shape[1]

R2_dff_all = []
R2_spikes_all = []

for roi_idx in range(num_rois):
    print(f"Processing ROI {roi_idx}")
    result = run_unit_rf_pipeline(mod, trace_type = trace_type, roi_idx=roi_idx)

    R2_dff = result['R2_dff']
    R2_spikes = result['R2_spikes']
   
    R2_dff_all.append(R2_dff)
    R2_spikes_all.append(R2_spikes)

In [ ]:
# histograms with median R2

import matplotlib.pyplot as plt

plt.hist(R2_dff_all, bins=20, alpha=0.5, label='R2_dff')
plt.hist(R2_spikes_all, bins=20, alpha=0.5, label='R2_spikes')
plt.axvline(x=np.median(R2_dff_all), color='blue', linestyle='--', label='Median R2_dff')
plt.axvline(x=np.median(R2_spikes_all), color='orange', linestyle='--', label='Median R2_spikes')
plt.xlabel('R2')
plt.ylabel('Frequency')
plt.legend()
plt.show()

# percent with R2 above 50%
threshold = 0.5
percent_R2_dff_above_thresh = 100 * np.sum(np.array(R2_dff_all) > threshold) / len(R2_dff_all)
percent_R2_spikes_above_thresh = 100 * np.sum(np.array(R2_spikes_all) > threshold) / len(R2_spikes_all)
print(f"Percent R2_dff above {threshold}: {percent_R2_dff_above_thresh:.2f}%")
print(f"Percent R2_spikes above {threshold}: {percent_R2_spikes_above_thresh:.2f}%")
